# v1.5.5 — PubMedBERT embedding of CTO criteria text (Colab GPU)

Run this notebook on Colab with a GPU runtime (Runtime → Change runtime type → T4 GPU).

**Inputs you provide:**
1. Upload `metadata_cache.jsonl` (the Stage 3 output, ~10-15 MB) via the file panel on the left.

**Outputs you download:**
- `cto_embeddings.npy` — [N, 768] float32 array, mean-pooled PubMedBERT embeddings of each trial's eligibility criteria text
- `cto_embeddings_nctid.txt` — row-index → NCT-ID map, one ID per line (alignment to embeddings.npy)

Drop the outputs into `api/data/cto/` on your local checkout. Stage 5 (combine HINT ∪ CTO + retrain) reads from there.

**Why PubMedBERT not BioBERT:** v1.5.2 swapped to PubMedBERT and AUC went from 0.6899 → 0.7030. The CTO benchmark must use the same embedding model the runtime engine consumes; otherwise the trained LightGBM would see features from a different distribution at training vs inference. See `methodology/09-ml-pos-prior.md` for the decision history.

## 1. Install dependencies

transformers + torch come pre-installed on most Colab runtimes; this pin avoids any version surprises.

In [ ]:
!pip install -q 'transformers>=4.46,<5' 'torch>=2.6,<3' 'huggingface-hub>=0.20,<1.0'
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Upload the metadata cache

If you don't see a file upload dialog, drag-and-drop `metadata_cache.jsonl` into the Colab file panel (left sidebar).

In [ ]:
import os, json
if not os.path.exists('metadata_cache.jsonl'):
    from google.colab import files
    print('Upload metadata_cache.jsonl (~10-15 MB)…')
    files.upload()

records = []
with open('metadata_cache.jsonl', 'r') as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

# Drop records with empty/None criteria (defense-in-depth — the runner
# already filters these but verifying here keeps the embedding pipeline
# tolerant of a slightly noisier upstream)
records = [r for r in records if r.get('eligibility_criteria')]
print(f'Loaded {len(records)} trials with non-empty criteria')
print(f'First trial: nct={records[0]["nct_id"]}, criteria_chars={len(records[0]["eligibility_criteria"])}')

## 3. Run PubMedBERT embeddings

Mean-pooled token embeddings (not CLS — see `api/app/modules/ml_pos_prior/bert_embed.py` docstring for the design choice). Wordpiece-truncation at 512 tokens, same as the runtime engine.

Batch size 32 fits comfortably in T4's 16 GB VRAM with PubMedBERT-base (110M params).

Expected wall-clock: ~30 min for 2,400 trials on a T4.

In [ ]:
import numpy as np
import torch
from transformers import AutoModel, AutoTokenizer

MODEL_ID = 'microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext'
MAX_LENGTH = 512
BATCH_SIZE = 32
EMBEDDING_DIM = 768
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Loading {MODEL_ID} on {device}…')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModel.from_pretrained(MODEL_ID).to(device).eval()
print('Model loaded')

criteria_texts = [r['eligibility_criteria'] for r in records]
nctids = [r['nct_id'] for r in records]

embeddings = np.zeros((len(criteria_texts), EMBEDDING_DIM), dtype=np.float32)

from time import time
started = time()
with torch.no_grad():
    for batch_start in range(0, len(criteria_texts), BATCH_SIZE):
        batch = criteria_texts[batch_start:batch_start + BATCH_SIZE]
        encoded = tokenizer(
            batch, return_tensors='pt', padding=True, truncation=True,
            max_length=MAX_LENGTH,
        ).to(device)
        out = model(**encoded)
        # Mean-pool over tokens, masking padding
        mask = encoded['attention_mask'].unsqueeze(-1).float()
        summed = (out.last_hidden_state * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1)
        pooled = (summed / counts).cpu().numpy()
        embeddings[batch_start:batch_start + len(batch)] = pooled

        if (batch_start // BATCH_SIZE) % 10 == 0:
            elapsed = time() - started
            done = batch_start + len(batch)
            rate = done / elapsed if elapsed > 0 else 0
            eta_min = (len(criteria_texts) - done) / rate / 60 if rate > 0 else 0
            print(f'{done}/{len(criteria_texts)} ({rate:.1f}/s, eta {eta_min:.1f}min)')

print(f'\nDone in {(time()-started)/60:.1f} min. Embeddings shape: {embeddings.shape}')

## 4. Save and download

Two files written:
- `cto_embeddings.npy` — the [N, 768] float32 array
- `cto_embeddings_nctid.txt` — the row-index → NCT-ID map (alignment to embeddings.npy)

Both are required as a pair — embeddings without the NCT-ID alignment is just noise.

In [ ]:
np.save('cto_embeddings.npy', embeddings)
with open('cto_embeddings_nctid.txt', 'w') as f:
    f.write('\n'.join(nctids) + '\n')

import os
print(f'cto_embeddings.npy: {os.path.getsize("cto_embeddings.npy") / 1024 / 1024:.2f} MB')
print(f'cto_embeddings_nctid.txt: {os.path.getsize("cto_embeddings_nctid.txt") / 1024:.2f} KB')

from google.colab import files
files.download('cto_embeddings.npy')
files.download('cto_embeddings_nctid.txt')

## 5. Sanity check

Quick correctness check before you close the notebook: similar criteria text (e.g. two oncology trials) should have higher cosine similarity than unrelated criteria (oncology vs CNS).

In [ ]:
from numpy.linalg import norm

def cosine(a, b):
    return float((a @ b) / (norm(a) * norm(b)))

# Find some labeled exemplars — first oncology + first CNS
onc_idx = next((i for i, r in enumerate(records) if 'cancer' in (r['eligibility_criteria'] or '').lower() or 'tumor' in (r['eligibility_criteria'] or '').lower()), None)
cns_idx = next((i for i, r in enumerate(records) if 'alzheimer' in (r['eligibility_criteria'] or '').lower() or 'parkinson' in (r['eligibility_criteria'] or '').lower() or 'depression' in (r['eligibility_criteria'] or '').lower()), None)

if onc_idx is None or cns_idx is None:
    print('Could not locate exemplars for sanity check; skipping')
else:
    # Find a second oncology trial different from the first
    onc2_idx = next((i for i in range(onc_idx+1, len(records)) if 'cancer' in (records[i]['eligibility_criteria'] or '').lower()), None)
    if onc2_idx is not None:
        sim_onc_onc = cosine(embeddings[onc_idx], embeddings[onc2_idx])
        sim_onc_cns = cosine(embeddings[onc_idx], embeddings[cns_idx])
        print(f'Oncology trial {records[onc_idx]["nct_id"]} ↔ oncology trial {records[onc2_idx]["nct_id"]} cosine sim: {sim_onc_onc:.3f}')
        print(f'Oncology trial {records[onc_idx]["nct_id"]} ↔ CNS trial {records[cns_idx]["nct_id"]} cosine sim: {sim_onc_cns:.3f}')
        print(f'Spread (onc-onc minus onc-cns): {sim_onc_onc - sim_onc_cns:.3f}')
        print('A positive spread of at least 0.02 indicates the embeddings are clustering meaningfully.')